0. Configuration

In [3]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
# Graine fixe pour obtenir les mêmes résultats à chaque exécution
GRAINE = 42
torch.manual_seed(GRAINE)
np.random.seed(GRAINE)

# Calcul sur la carte graphique si elle existe, sinon sur le processeur
appareil = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil utilisé :", appareil)

# Paramètres de l'entraînement
TAILLE_LOT = 4096
NOMBRE_ESSAIS_RECHERCHE = 6      # essais de la recherche aléatoire, par modèle
EPOQUES_RECHERCHE = 10           # époques max pendant la recherche
EPOQUES_FINALES = 30             # époques max pour l'entraînement final
PATIENCE = 5                     # early stopping : on arrête après 5 époques sans amélioration
GRAINES_FINALES = [0, 1, 2]      # on entraîne 3 fois avec des graines différentes


ModuleNotFoundError: No module named 'torch'

1. Chargement des données

In [ ]:
transactions = pd.read_csv("credit_card_data.csv")

print("Forme :", transactions.shape)
print("Valeurs manquantes :", transactions.isna().sum().sum())
print(f"Taux de fraude : {transactions['fraud'].mean() * 100:.2f} %")
transactions.head()
# Moyenne de chaque variable pour les transactions normales (0) et frauduleuses (1)
transactions.groupby("fraud").mean().round(2)

Forme : (1000000, 8)
Valeurs manquantes : 0
Taux de fraude : 8.74 %


,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order
fraud,,,,,,,
0.0,22.83,4.30,1.42,0.88,0.36,0.11,0.62
1.0,66.26,12.71,6.01,0.88,0.26,0.00,0.95


2. Prétraitement

In [ ]:
# Les 7 colonnes d'entrée (la colonne "fraud" n'est PAS une entrée du modèle)
COLONNES_CONTINUES = ["distance_from_home", "distance_from_last_transaction", "ratio_to_median_purchase_price"]
COLONNES_BINAIRES = ["repeat_retailer", "used_chip", "used_pin_number", "online_order"]
COLONNES_ENTREE = COLONNES_CONTINUES + COLONNES_BINAIRES

3. Découpage des données (entraînement / validation / test)

In [ ]:
# On sépare les transactions normales et frauduleuses, puis on les mélange
transactions_normales = transactions[transactions["fraud"] == 0].sample(frac=1, random_state=GRAINE)
transactions_frauduleuses = transactions[transactions["fraud"] == 1].sample(frac=1, random_state=GRAINE)

# Les transactions normales sont découpées : 80 % entraînement, 10 % validation, 10 % test
nombre_normales = len(transactions_normales)
fin_entrainement = int(0.8 * nombre_normales)
fin_validation = int(0.9 * nombre_normales)

normales_entrainement = transactions_normales.iloc[:fin_entrainement]
normales_validation = transactions_normales.iloc[fin_entrainement:fin_validation]
normales_test = transactions_normales.iloc[fin_validation:]

# On ajoute des fraudes à la validation et au test, en gardant le taux de fraude du jeu de données
taux_fraude = transactions["fraud"].mean()
nombre_fraudes = int(len(normales_validation) * taux_fraude / (1 - taux_fraude))
fraudes_validation = transactions_frauduleuses.iloc[:nombre_fraudes]
fraudes_test = transactions_frauduleuses.iloc[nombre_fraudes:2 * nombre_fraudes]

print("Entraînement :", len(normales_entrainement), "normales, 0 fraude")
print("Validation   :", len(normales_validation), "normales,", len(fraudes_validation), "fraudes")
print("Test         :", len(normales_test), "normales,", len(fraudes_test), "fraudes")

Entraînement : 730077 normales, 0 fraude
Validation   : 91260 normales, 8740 fraudes
Test         : 91260 normales, 8740 fraudes


4. Transformation des données (log + standardisation)

In [ ]:
# Les distances et le ratio ont de très grandes valeurs : le log les rend plus régulières
def appliquer_log(tableau):
    entrees = tableau[COLONNES_ENTREE].copy()
    entrees[COLONNES_CONTINUES] = np.log1p(entrees[COLONNES_CONTINUES])
    return entrees.values.astype("float32")

# La moyenne et l'écart-type viennent de l'entraînement seulement (pas de fuite de données)
entrees_entrainement_log = appliquer_log(normales_entrainement)
moyenne = entrees_entrainement_log.mean(axis=0)
ecart_type = entrees_entrainement_log.std(axis=0)

# Applique le log puis la standardisation, et retourne un tenseur PyTorch
def preparer(tableau):
    return torch.tensor((appliquer_log(tableau) - moyenne) / ecart_type)

entrees_entrainement = preparer(normales_entrainement)
entrees_validation_normales = preparer(normales_validation)
entrees_validation = preparer(pd.concat([normales_validation, fraudes_validation]))
entrees_test = preparer(pd.concat([normales_test, fraudes_test]))

# Étiquettes (0 = normale, 1 = fraude) : servent seulement à évaluer, jamais à entraîner
etiquettes_validation = np.r_[np.zeros(len(normales_validation)), np.ones(len(fraudes_validation))]
etiquettes_test = np.r_[np.zeros(len(normales_test)), np.ones(len(fraudes_test))]

In [ ]:
# Les données sont lues par lots. La cible d'un auto-encodeur est l'entrée elle-même (pas d'étiquette)
chargeur_entrainement = DataLoader(TensorDataset(entrees_entrainement), batch_size=TAILLE_LOT, shuffle=True)
chargeur_validation = DataLoader(TensorDataset(entrees_validation_normales), batch_size=TAILLE_LOT)

# Petit sous-ensemble pour que la recherche d'hyperparamètres soit rapide
entrees_recherche = entrees_entrainement[:100000]
chargeur_recherche = DataLoader(TensorDataset(entrees_recherche), batch_size=TAILLE_LOT, shuffle=True)

5. Le modèle : auto-encodeur dense

In [ ]:
class AutoEncodeur(nn.Module):
    def __init__(self, taille_latente, taille_cachee):
        super().__init__()
        # Encodeur : 7 variables -> couche cachée -> code (taille_latente)
        self.encodeur = nn.Sequential(
            nn.Linear(7, taille_cachee),
            nn.ReLU(),                                # ReLU : simple et efficace pour les couches cachées
            nn.Linear(taille_cachee, taille_latente),  # pas d'activation : le code peut être négatif
        )
        # Décodeur : miroir de l'encodeur, il reconstruit les 7 variables
        self.decodeur = nn.Sequential(
            nn.Linear(taille_latente, taille_cachee),
            nn.ReLU(),
            nn.Linear(taille_cachee, 7),               # pas d'activation : les données standardisées peuvent être négatives
        )

    def forward(self, entrees):
        return self.decodeur(self.encodeur(entrees))

In [ ]:
# Test rapide : le modèle prend 7 variables et en redonne 7
modele_test = AutoEncodeur(taille_latente=3, taille_cachee=16)
lot_exemple = entrees_entrainement[:5]
print("Taille du code :", modele_test.encodeur(lot_exemple).shape)
print("Taille de la sortie :", modele_test(lot_exemple).shape)

Taille du code : torch.Size([5, 3])
Taille de la sortie : torch.Size([5, 7])


6. Fonctions d'entraînement et d'évaluation

In [ ]:
fonction_perte = nn.MSELoss()   # erreur quadratique moyenne entre l'entrée et la reconstruction

def entrainer(modele, taux_apprentissage, nombre_epoques, patience, chargeur, bruit=0.0):
    # Entraîne le modèle avec early stopping et redonne les meilleurs poids
    modele.to(appareil)
    optimiseur = torch.optim.Adam(modele.parameters(), lr=taux_apprentissage)
    pertes_entrainement, pertes_validation = [], []
    meilleure_perte = float("inf")
    meilleurs_poids = None
    meilleure_epoque = 0
    epoques_sans_amelioration = 0

    for epoque in range(nombre_epoques):
        # --- Entraînement ---
        modele.train()
        perte_totale = 0.0
        for (lot,) in chargeur:
            lot = lot.to(appareil)
            lot_bruite = lot + bruit * torch.randn_like(lot)   # bruit = 0 pour le modèle classique
            optimiseur.zero_grad()
            perte = fonction_perte(modele(lot_bruite), lot)    # on compare à l'entrée PROPRE
            perte.backward()
            optimiseur.step()
            perte_totale += perte.item()
        pertes_entrainement.append(perte_totale / len(chargeur))

        # --- Validation (transactions normales seulement) ---
        modele.eval()
        perte_totale = 0.0
        with torch.no_grad():
            for (lot,) in chargeur_validation:
                lot = lot.to(appareil)
                perte_totale += fonction_perte(modele(lot), lot).item()
        pertes_validation.append(perte_totale / len(chargeur_validation))

        # --- Early stopping : on garde les meilleurs poids ---
        if pertes_validation[-1] < meilleure_perte:
            meilleure_perte = pertes_validation[-1]
            meilleurs_poids = copy.deepcopy(modele.state_dict())
            meilleure_epoque = epoque
            epoques_sans_amelioration = 0
        else:
            epoques_sans_amelioration += 1
            if epoques_sans_amelioration >= patience:
                break

    modele.load_state_dict(meilleurs_poids)
    modele.eval()
    return pertes_entrainement, pertes_validation, meilleure_epoque

In [ ]:
def calculer_erreurs(modele, entrees):
    # Erreur de reconstruction de CHAQUE transaction (moyenne des erreurs au carré sur les 7 variables)
    modele.to(appareil).eval()
    with torch.no_grad():
        entrees = entrees.to(appareil)
        reconstruction = modele(entrees)
        return ((reconstruction - entrees) ** 2).mean(dim=1).cpu().numpy()

def score_validation(modele):
    # Qualité du modèle sur la validation (normales + fraudes), mesurée par la PR-AUC
    erreurs = calculer_erreurs(modele, entrees_validation)
    return average_precision_score(etiquettes_validation, erreurs)

7. Recherche d'hyperparamètres (recherche aléatoire)

In [ ]:
generateur_aleatoire = np.random.default_rng(GRAINE)
meilleurs_parametres = {}

for type_modele in ["classique", "debruitant"]:
    meilleur_score = -1
    for essai in range(NOMBRE_ESSAIS_RECHERCHE):
        # On tire au hasard des hyperparamètres
        parametres = {
            "taille_latente": int(generateur_aleatoire.integers(2, 6)),
            "taille_cachee": int(generateur_aleatoire.choice([8, 16, 32, 64])),
            "taux_apprentissage": float(10 ** generateur_aleatoire.uniform(-3.5, -2)),
            "bruit": float(generateur_aleatoire.uniform(0.05, 0.5)) if type_modele == "debruitant" else 0.0,
        }

        # On entraîne un modèle court avec ces hyperparamètres, puis on le note
        torch.manual_seed(GRAINE)
        modele = AutoEncodeur(parametres["taille_latente"], parametres["taille_cachee"])
        entrainer(modele, parametres["taux_apprentissage"], EPOQUES_RECHERCHE, 3,
                  chargeur_recherche, bruit=parametres["bruit"])
        score = score_validation(modele)

        if score > meilleur_score:
            meilleur_score = score
            meilleurs_parametres[type_modele] = parametres

    print(f"{type_modele:10s} meilleure PR-AUC validation = {meilleur_score:.4f}")
    print("           ", meilleurs_parametres[type_modele])

classique  meilleure PR-AUC validation = 0.5746
            {'taille_latente': 2, 'taille_cachee': 64, 'taux_apprentissage': 0.0014398513556508027, 'bruit': 0.0}
debruitant meilleure PR-AUC validation = 0.3058
            {'taille_latente': 5, 'taille_cachee': 64, 'taux_apprentissage': 0.002802183675378178, 'bruit': 0.3911394830384182}


8. Entraînement final des deux modèles

In [ ]:
resultats = {}

for type_modele, parametres in meilleurs_parametres.items():
    scores, modeles, courbes = [], [], []

    # On entraîne 3 fois (graines différentes) : le hasard change un peu le résultat
    for graine in GRAINES_FINALES:
        torch.manual_seed(graine)
        modele = AutoEncodeur(parametres["taille_latente"], parametres["taille_cachee"])
        courbe = entrainer(modele, parametres["taux_apprentissage"], EPOQUES_FINALES, PATIENCE,
                           chargeur_entrainement, bruit=parametres["bruit"])
        scores.append(score_validation(modele))
        modeles.append(modele)
        courbes.append(courbe)

    # On garde la meilleure des 3 graines
    meilleur_indice = int(np.argmax(scores))
    resultats[type_modele] = {"modele": modeles[meilleur_indice], "courbe": courbes[meilleur_indice],
                              "scores": scores, "parametres": parametres}
    print(f"{type_modele:10s} PR-AUC validation : {np.mean(scores):.4f} ± {np.std(scores):.4f}")

In [ ]:
# Courbes d'apprentissage : la perte doit baisser, sans que la validation remonte
figure, axes = plt.subplots(1, 2, figsize=(11, 4))
for axe, (type_modele, resultat) in zip(axes, resultats.items()):
    pertes_entrainement, pertes_validation, meilleure_epoque = resultat["courbe"]
    axe.plot(pertes_entrainement, label="Entraînement")
    axe.plot(pertes_validation, label="Validation")
    axe.axvline(meilleure_epoque, color="gray", linestyle="--", label="Meilleure époque")
    axe.set_title(type_modele)
    axe.set_xlabel("Époque")
    axe.set_ylabel("Perte (MSE)")
    axe.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Modèle final = celui qui a la meilleure PR-AUC moyenne sur la validation
modele_retenu = max(resultats, key=lambda nom: np.mean(resultats[nom]["scores"]))
modele_final = resultats[modele_retenu]["modele"]
parametres_finaux = resultats[modele_retenu]["parametres"]
print("Modèle retenu :", modele_retenu)
print(parametres_finaux)

9. Seuil de détection

In [ ]:
POURCENTILES_TESTES = [90, 93, 95, 97, 98, 99]
RAPPEL_MINIMUM = 0.75   # objectif : détecter au moins 75 % des fraudes

def choisir_seuil(modele):
    # Teste plusieurs percentiles et garde le plus haut qui respecte le rappel minimum (sur la validation)
    erreurs_validation = calculer_erreurs(modele, entrees_validation)
    erreurs_normales = erreurs_validation[etiquettes_validation == 0]
    lignes = []
    for percentile in POURCENTILES_TESTES:
        seuil_essai = np.percentile(erreurs_normales, percentile)
        predictions = erreurs_validation > seuil_essai
        lignes.append({"percentile": percentile,
                       "seuil": seuil_essai,
                       "précision": precision_score(etiquettes_validation, predictions),
                       "rappel": recall_score(etiquettes_validation, predictions),
                       "F1": f1_score(etiquettes_validation, predictions)})
    tableau = pd.DataFrame(lignes)
    tableau_valide = tableau[tableau["rappel"] >= RAPPEL_MINIMUM]
    if len(tableau_valide) > 0:
        meilleure_ligne = tableau_valide.iloc[-1]      # le plus haut percentile = le moins de fausses alertes
    else:
        meilleure_ligne = tableau.loc[tableau["F1"].idxmax()]
    return float(meilleure_ligne["seuil"]), int(meilleure_ligne["percentile"]), tableau

In [ ]:
# Seuil du modèle final
seuil, percentile_seuil, tableau_seuils = choisir_seuil(modele_final)
print(tableau_seuils.round(3).to_string(index=False))
print(f"\nPercentile retenu : {percentile_seuil} (le plus haut qui garde un rappel d'au moins 75 % sur la validation)")
print(f"Seuil de détection : {seuil:.4f}")

10. Évaluation sur le jeu de test

In [ ]:
def evaluer(modele, seuil_modele):
    erreurs = calculer_erreurs(modele, entrees_test)
    predictions = erreurs > seuil_modele
    mesures = {
        "PR-AUC": average_precision_score(etiquettes_test, erreurs),
        "ROC-AUC": roc_auc_score(etiquettes_test, erreurs),
        "précision": precision_score(etiquettes_test, predictions),
        "rappel": recall_score(etiquettes_test, predictions),
        "F1": f1_score(etiquettes_test, predictions),
        "fausses alertes": predictions[etiquettes_test == 0].mean(),
    }
    return mesures, erreurs, predictions

# Comparaison des deux modèles (chacun avec son propre seuil, choisi sur la validation)
comparaison = {}
for type_modele, resultat in resultats.items():
    seuil_modele, _, _ = choisir_seuil(resultat["modele"])
    comparaison[type_modele] = evaluer(resultat["modele"], seuil_modele)[0]

print(pd.DataFrame(comparaison).T.round(4).to_string())
print(f"\nRéférence : un modèle au hasard aurait une PR-AUC de {etiquettes_test.mean():.4f}")

# Résultats détaillés du modèle final
mesures, erreurs_test, predictions_test = evaluer(modele_final, seuil)
print(f"=== Modèle final ({modele_retenu}) ===")
for nom_mesure, valeur in mesures.items():
    print(f"{nom_mesure:16s} {valeur:.4f}")

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# 1) Distribution de l'erreur de reconstruction
intervalles = np.logspace(np.log10(erreurs_test.min() + 1e-6), np.log10(erreurs_test.max()), 60)
axes[0].hist(erreurs_test[etiquettes_test == 0], bins=intervalles, alpha=0.6, label="Normales")
axes[0].hist(erreurs_test[etiquettes_test == 1], bins=intervalles, alpha=0.6, label="Fraudes")
axes[0].axvline(seuil, color="black", linestyle="--", label="Seuil")
axes[0].set_xscale("log")
axes[0].set_xlabel("Erreur de reconstruction (échelle log)")
axes[0].set_ylabel("Nombre de transactions")
axes[0].set_title("Distribution de l'erreur")
axes[0].legend()

# 2) Courbe précision-rappel
precisions, rappels, _ = precision_recall_curve(etiquettes_test, erreurs_test)
axes[1].plot(rappels, precisions, label=f"PR-AUC = {mesures['PR-AUC']:.3f}")
axes[1].scatter([mesures["rappel"]], [mesures["précision"]], color="red", zorder=3, label="Seuil retenu")
axes[1].set_xlabel("Rappel")
axes[1].set_ylabel("Précision")
axes[1].set_title("Courbe précision-rappel")
axes[1].legend()

# 3) Matrice de confusion
ConfusionMatrixDisplay(confusion_matrix(etiquettes_test, predictions_test),
                       display_labels=["Normale", "Fraude"]).plot(ax=axes[2], cmap="Blues", colorbar=False)
axes[2].set_title("Matrice de confusion")

plt.tight_layout()
plt.show()

11. Exemples concrets de fraudes détectées

In [ ]:
def reconstruire(modele, entrees):
    # Retourne la transaction originale et reconstruite, dans leurs unités d'origine
    modele.eval()
    with torch.no_grad():
        reconstruction = modele(entrees.to(appareil)).cpu().numpy()
    def remettre_en_unites(valeurs):
        valeurs = valeurs * ecart_type + moyenne
        valeurs[:, :3] = np.expm1(valeurs[:, :3])      # inverse du log
        return valeurs
    return remettre_en_unites(entrees.numpy().copy()), remettre_en_unites(reconstruction)

# Les 3 fraudes détectées avec la plus grande erreur
fraudes_detectees = np.where((etiquettes_test == 1) & predictions_test)[0]
indices_exemples = fraudes_detectees[np.argsort(-erreurs_test[fraudes_detectees])][:3]
originales, reconstruites = reconstruire(modele_final, entrees_test[indices_exemples])

for position, indice in enumerate(indices_exemples):
    print(f"--- Fraude n°{indice} | erreur = {erreurs_test[indice]:.2f} (seuil = {seuil:.3f}) ---")
    tableau = pd.DataFrame({"original": originales[position], "reconstruit": reconstruites[position]},
                           index=COLONNES_ENTREE)
    print(tableau.round(2).T.to_string(), "\n")

In [ ]:
# Quelles variables expliquent l'erreur ? (interprétabilité)
# On calcule l'erreur de chaque variable, pour les normales, les fraudes détectées et les fraudes manquées
def erreur_par_variable(indices):
    entrees = entrees_test[indices]
    with torch.no_grad():
        reconstruction = modele_final(entrees.to(appareil)).cpu()
    return ((reconstruction - entrees) ** 2).mean(dim=0).numpy()

indices_normales = np.where(etiquettes_test == 0)[0][:5000]
indices_fraudes_manquees = np.where((etiquettes_test == 1) & ~predictions_test)[0]

positions = np.arange(7)
largeur = 0.28
plt.figure(figsize=(10, 4))
plt.bar(positions - largeur, erreur_par_variable(indices_normales), largeur, label="Normales")
plt.bar(positions, erreur_par_variable(fraudes_detectees), largeur, label="Fraudes détectées")
plt.bar(positions + largeur, erreur_par_variable(indices_fraudes_manquees), largeur, label="Fraudes manquées")
plt.xticks(positions, COLONNES_ENTREE, rotation=25, ha="right")
plt.ylabel("Erreur moyenne (variables standardisées)")
plt.title("Quelles variables causent l'erreur ?")
plt.legend()
plt.tight_layout()
plt.show()

12. Sauvegarde et rechargement du modèle

In [ ]:
# On sauvegarde les poids ET tout ce qu'il faut pour réutiliser le modèle (seuil, moyenne, écart-type)
torch.save({
    "poids": modele_final.cpu().state_dict(),
    "taille_latente": parametres_finaux["taille_latente"],
    "taille_cachee": parametres_finaux["taille_cachee"],
    "moyenne": moyenne.tolist(),
    "ecart_type": ecart_type.tolist(),
    "seuil": seuil,
}, "modele_final.pt")
print("Modèle sauvegardé : modele_final.pt")

In [ ]:
# Rechargement dans un modèle neuf
sauvegarde = torch.load("modele_final.pt", map_location=appareil, weights_only=True)
modele_recharge = AutoEncodeur(sauvegarde["taille_latente"], sauvegarde["taille_cachee"]).to(appareil)
modele_recharge.load_state_dict(sauvegarde["poids"])
modele_recharge.eval()

# Vérification : le modèle rechargé donne les mêmes erreurs que l'original
modele_final.to(appareil)
erreurs_original = calculer_erreurs(modele_final, entrees_test[:1000])
erreurs_recharge = calculer_erreurs(modele_recharge, entrees_test[:1000])
print("Modèle rechargé identique à l'original :", np.allclose(erreurs_original, erreurs_recharge, atol=1e-5))